# Malicious URL Detector — Exploratory Notebook
Use this to experiment with features, visualise data, and prototype new models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_extraction import extract_features, FEATURE_COLUMNS
%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
# Load dataset
df = pd.read_csv('../data/urls_dataset.csv')
print(df.shape)
df.head()

In [ ]:
# Class distribution
df['label'].value_counts().plot(kind='bar', color=['steelblue','firebrick'])
plt.title('Class Distribution')
plt.xticks([0,1], ['Benign','Malicious'], rotation=0)
plt.show()

In [ ]:
# Feature matrix
features = [extract_features(u) for u in df['url']]
feat_df = pd.DataFrame(features)[FEATURE_COLUMNS]
feat_df['label'] = df['label'].values
feat_df.head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14,10))
corr = feat_df.corr()
sns.heatmap(corr, annot=False, fmt='.2f', cmap='coolwarm', linewidths=0.3)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# URL length by class
fig, axes = plt.subplots(1,2, figsize=(12,4))
for i, cls in enumerate([0,1]):
    feat_df[feat_df['label']==cls]['url_length'].hist(
        bins=30, ax=axes[i], color='steelblue' if cls==0 else 'firebrick', alpha=0.7)
    axes[i].set_title(f"URL Length — {'Benign' if cls==0 else 'Malicious'}")
plt.tight_layout()
plt.show()

In [ ]:
# Load and test saved model
import joblib
payload = joblib.load('../models/best_model.joblib')
model = payload['model']

test_urls = [
    'https://www.google.com/',
    'http://paypal.com.secure-verify.xyz/login',
    'http://192.168.1.1/admin'
]
for u in test_urls:
    f = extract_features(u)
    X = np.array([f.get(c,0) for c in FEATURE_COLUMNS]).reshape(1,-1)
    prob = model.predict_proba(X)[0]
    print(f"{u[:60]:60s}  safe={prob[0]:.2%}  malicious={prob[1]:.2%}")